# W1 Day-B: Log Parsing with Drain3 and Anomaly Detection
## Assignment Notebook

This notebook uses real Loghub datasets copied into `w1/day-b/data` for a self-contained submission.

- Primary dataset for Phases 1-3: `data/BGL_2k.log`
- Secondary dataset for Phase 4 comparison: `data/HDFS_2k.log`

Reason for choosing BGL as the primary dataset: the raw BGL log contains alert labels in the first column, so we can evaluate anomaly detection directly on real data.


In [1]:
import sys
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity

try:
    from drain3.drain import Drain
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "drain3"])
    from drain3.drain import Drain

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("deep")

ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
SCRIPTS_DIR = ROOT / "scripts"
RESULTS_DIR.mkdir(exist_ok=True)

BGL_LOG = DATA_DIR / "BGL_2k.log"
HDFS_LOG = DATA_DIR / "HDFS_2k.log"

if not BGL_LOG.exists():
    raise FileNotFoundError(f"Missing BGL dataset: {BGL_LOG}")
if not HDFS_LOG.exists():
    raise FileNotFoundError(f"Missing HDFS dataset: {HDFS_LOG}")

def normalize_result(result):
    if isinstance(result, tuple):
        cluster, change_type = result
        return int(cluster.cluster_id), cluster.get_template(), change_type
    return int(result.cluster_id), result.get_template(), None

def extract_timestamp(log_line):
    parts = log_line.split()
    if len(parts) >= 5:
        try:
            return pd.to_datetime(parts[4], format="%Y-%m-%d-%H.%M.%S.%f", errors="raise")
        except (ValueError, TypeError):
            pass
    if len(parts) >= 2:
        try:
            return pd.to_datetime(f"{parts[0]} {parts[1]}", format="%y%m%d %H%M%S", errors="raise")
        except (ValueError, TypeError):
            pass
        try:
            return pd.to_datetime(f"{parts[0]} {parts[1]}", format="%Y-%m-%d %H:%M:%S", errors="raise")
        except (ValueError, TypeError):
            pass
    return pd.NaT

def bgl_is_alert(log_line):
    parts = log_line.split(maxsplit=1)
    if not parts:
        return 0
    return 0 if parts[0] == "-" else 1

def load_lines(path):
    return [line.strip() for line in path.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]


## Dataset Load

Use the real `BGL_2k.log` copied from Loghub into `data/` as the main dataset. This keeps the submission self-contained and still uses real data with alert labels directly from the raw log format.


In [2]:
primary_dataset_name = "BGL"
primary_log_path = BGL_LOG
comparison_log_path = HDFS_LOG

logs = load_lines(primary_log_path)
df_labels = pd.DataFrame(
    {
        "log_id": range(len(logs)),
        "is_alert": [bgl_is_alert(line) for line in logs],
    }
)

print(f"Primary dataset: {primary_dataset_name}")
print(f"Log path: {primary_log_path}")
print(f"Total log lines: {len(logs):,}")
print(f"Alert logs: {int(df_labels['is_alert'].sum()):,}")
print("First 5 log entries:")
for idx, log in enumerate(logs[:5], 1):
    print(f"  {idx}. {log}")


Primary dataset: BGL
Log path: C:\Users\Admin\aiops-Thinh\w1\day-b\data\BGL_2k.log
Total log lines: 2,000
Alert logs: 143
First 5 log entries:
  1. - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.675872 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
  2. - 1117838573 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.53.276129 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
  3. - 1117838976 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.49.36.156884 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
  4. - 1117838978 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.49.38.026704 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
  5. - 1117842440 2005.06.03 R23-M0-NE-C:J05-U01 2005-06-03-16.47.20.730545 R23-M0-NE-C:J05-U01 RAS KERNEL INFO 63543 double-hummer alignment exceptions


## Phase 1: Parse Log with Drain3

In [3]:
drain = Drain(sim_th=0.5, max_children=100, max_clusters=None)
parsed_rows = []

for idx, log in enumerate(logs):
    cluster_id, template, change_type = normalize_result(drain.add_log_message(log))
    parsed_rows.append(
        {
            "log_id": idx,
            "original_log": log,
            "template_id": cluster_id,
            "template": template,
            "change_type": change_type,
            "timestamp": extract_timestamp(log),
            "is_alert": bgl_is_alert(log),
        }
    )

df_parsed = pd.DataFrame(parsed_rows)
template_map = {int(cluster.cluster_id): cluster.get_template() for cluster in drain.clusters}
template_counts = df_parsed.groupby("template_id", as_index=False).size().rename(columns={"size": "count"})
template_counts["template"] = template_counts["template_id"].map(template_map)
template_counts = template_counts.sort_values(["count", "template_id"], ascending=[False, True])
template_counts[["template_id", "template", "count"]].head(10).to_csv(RESULTS_DIR / "top_templates.csv", index=False)

print(f"Parsed rows: {len(df_parsed):,}")
print(f"Unique templates: {len(template_counts)}")
print(template_counts.head(10).to_string(index=False))
print(f"Saved top templates to {RESULTS_DIR / 'top_templates.csv'}")


Parsed rows: 2,000
Unique templates: 151
 template_id  count                                                                                                                                                                                                        template
          73    180                                                                                                                                                     - <*> 2005.07.09 <*> <*> <*> RAS KERNEL INFO generating <*>
          85    121                                                                                                                                   - <*> <*> <*> <*> <*> RAS KERNEL INFO <*> floating point alignment exceptions
           2    109                                                                                                                                    - <*> <*> <*> <*> <*> RAS KERNEL INFO <*> double-hummer alignment exceptions
           3     92                            

In [4]:
similarity_thresholds = [0.3, 0.5, 0.7]
tuning_rows = []

for sim_th in similarity_thresholds:
    tuned = Drain(sim_th=sim_th, max_children=100, max_clusters=None)
    for log in logs:
        tuned.add_log_message(log)
    tuning_rows.append(
        {
            "similarity_threshold": sim_th,
            "num_templates": len(tuned.clusters),
            "avg_cluster_size": round(len(logs) / len(tuned.clusters), 2),
        }
    )

df_tuning = pd.DataFrame(tuning_rows)
df_tuning.to_csv(RESULTS_DIR / "tuning_results.csv", index=False)
best_threshold = 0.5

print(df_tuning.to_string(index=False))
print(f"Selected sim_th: {best_threshold}")


 similarity_threshold  num_templates  avg_cluster_size
                  0.3             73             27.40
                  0.5            151             13.25
                  0.7           1459              1.37
Selected sim_th: 0.5


## Phase 2: Template Count Time Series and Anomaly Detection

In [5]:
df_parsed = df_parsed[df_parsed["timestamp"].notna()].copy()
df_parsed["time_window"] = df_parsed["timestamp"].dt.floor("30min")

template_counts_ts = df_parsed.groupby(["time_window", "template_id"]).size().reset_index(name="count")
window_template_ts = (
    template_counts_ts.groupby("time_window")["template_id"]
    .nunique()
    .reset_index(name="unique_template_count")
)
window_labels = df_parsed.groupby("time_window")["is_alert"].max().reset_index(name="label")

counts = window_template_ts["unique_template_count"].to_numpy()
mean_count = counts.mean()
std_count = counts.std()
upper_threshold = mean_count + 3 * std_count
lower_threshold = max(0, mean_count - 3 * std_count)
window_template_ts["is_anomaly_3sigma"] = (
    (window_template_ts["unique_template_count"] > upper_threshold)
    | (window_template_ts["unique_template_count"] < lower_threshold)
)

iso_forest = IsolationForest(contamination=0.1, random_state=42)
window_template_ts["is_anomaly_if"] = iso_forest.fit_predict(window_template_ts[["unique_template_count"]]) == -1

labels_aligned = window_template_ts.merge(window_labels, on="time_window", how="left").fillna({"label": 0})
labels_aligned["label"] = labels_aligned["label"].astype(int)

precision_3sigma = precision_score(labels_aligned["label"], labels_aligned["is_anomaly_3sigma"], zero_division=0)
recall_3sigma = recall_score(labels_aligned["label"], labels_aligned["is_anomaly_3sigma"], zero_division=0)
f1_3sigma = f1_score(labels_aligned["label"], labels_aligned["is_anomaly_3sigma"], zero_division=0)

precision_if = precision_score(labels_aligned["label"], labels_aligned["is_anomaly_if"], zero_division=0)
recall_if = recall_score(labels_aligned["label"], labels_aligned["is_anomaly_if"], zero_division=0)
f1_if = f1_score(labels_aligned["label"], labels_aligned["is_anomaly_if"], zero_division=0)

spike_templates = []
for template_id in template_counts_ts["template_id"].unique():
    per_template = template_counts_ts[template_counts_ts["template_id"] == template_id].copy()
    template_mean = per_template["count"].mean()
    template_std = per_template["count"].std()
    if pd.isna(template_std):
        template_std = 0.0
    spike_windows = per_template[per_template["count"] > template_mean + 2 * template_std]
    if not spike_windows.empty:
        spike_templates.append(
            {
                "template_id": template_id,
                "template": template_map[template_id],
                "spike_windows": len(spike_windows),
                "max_count": int(per_template["count"].max()),
                "avg_count": round(template_mean, 2),
            }
        )
spike_df = pd.DataFrame(spike_templates).sort_values(["spike_windows", "max_count"], ascending=[False, False]) if spike_templates else pd.DataFrame()

split_idx = int(len(df_parsed) * 0.9)
early_templates = set(df_parsed.iloc[:split_idx]["template_id"].unique())
late_templates = set(df_parsed.iloc[split_idx:]["template_id"].unique())
new_templates = sorted(late_templates - early_templates)

print(f"Total windows: {len(window_template_ts)}")
print(f"Window size: 30 minutes")
print(f"Baseline series: unique template count per window")
print(f"3σ anomalies: {int(window_template_ts['is_anomaly_3sigma'].sum())}")
print(f"Isolation Forest anomalies: {int(window_template_ts['is_anomaly_if'].sum())}")
print()
print("Evaluation against BGL alert labels:")
print(f"  3σ precision={precision_3sigma:.3f}, recall={recall_3sigma:.3f}, f1={f1_3sigma:.3f}")
print(f"  IF precision={precision_if:.3f}, recall={recall_if:.3f}, f1={f1_if:.3f}")
print()
print("Spike templates:")
if spike_df.empty:
    print("  None")
else:
    print(spike_df[["template_id", "spike_windows", "avg_count", "max_count"]].head(10).to_string(index=False))
print()
print(f"New templates in final 10% of logs: {len(new_templates)}")
for template_id in new_templates[:5]:
    print(f"  Template {template_id}: {template_map[template_id]}")


Total windows: 527
Window size: 30 minutes
Baseline series: unique template count per window
3σ anomalies: 4
Isolation Forest anomalies: 12

Evaluation against BGL alert labels:
  3σ precision=0.500, recall=0.035, f1=0.066
  IF precision=0.167, recall=0.035, f1=0.058

Spike templates:
 template_id  spike_windows  avg_count  max_count
           3              8       1.10          2
          85              2       4.65         23
           2              2       4.54         15
         104              2       1.43          3
          73              1       9.47         40
          14              1       5.00         10
           1              1       1.91          7
          10              1       1.50          5
          49              1       1.62          4
          95              1       1.12          2

New templates in final 10% of logs: 15
  Template 137: - <*> 2005.12.01 <*> <*> <*> RAS KERNEL INFO 0 microseconds spent in the rbs signal handler during 0 calls. 

In [6]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    window_template_ts["time_window"],
    window_template_ts["unique_template_count"],
    marker="o",
    linewidth=1.2,
    label="Unique template count",
)
ax.scatter(
    window_template_ts.loc[window_template_ts["is_anomaly_3sigma"], "time_window"],
    window_template_ts.loc[window_template_ts["is_anomaly_3sigma"], "unique_template_count"],
    color="red",
    s=90,
    marker="X",
    label="3σ anomaly",
    zorder=5,
)
ax.axhline(mean_count, color="green", linestyle="--", alpha=0.7, label=f"Mean ({mean_count:.1f})")
ax.axhline(upper_threshold, color="red", linestyle="--", alpha=0.7, label=f"Upper 3σ ({upper_threshold:.1f})")
ax.set_title("BGL Unique Template Count Time Series with Highlighted Anomalies")
ax.set_xlabel("Time window")
ax.set_ylabel("Unique template count")
ax.legend(loc="best")
plt.xticks(rotation=45)
plt.tight_layout()
plot_path = RESULTS_DIR / "template_count_timeseries.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved plot to {plot_path}")


Saved plot to C:\Users\Admin\aiops-Thinh\w1\day-b\results\template_count_timeseries.png


assignment.ipynb#cell6:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Phase 3: Embedding, Similarity, and Novel Log Injection

In [7]:
top_n = min(20, len(template_counts))
top_templates = template_counts.head(top_n)["template"].tolist()
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 3), max_features=150)
tfidf_matrix = vectorizer.fit_transform(top_templates)
similarity_matrix = cosine_similarity(tfidf_matrix)

clusters_sim = []
visited = set()
for i in range(top_n):
    if i in visited:
        continue
    cluster = [i]
    for j in range(i + 1, top_n):
        if similarity_matrix[i, j] >= 0.7:
            cluster.append(j)
    if len(cluster) > 1:
        clusters_sim.append(cluster)
        visited.update(cluster)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Mean similarity: {similarity_matrix.mean():.3f}")
print(f"Clusters found above 0.7 similarity: {len(clusters_sim)}")
for idx, cluster in enumerate(clusters_sim[:3], 1):
    print(f"\nCluster {idx}")
    for item in cluster:
        print(f"  - {top_templates[item]}")

anomalous_log = "GPUFAIL 1119999999 2005.06.20 R99-M9-N9-C:J99-U99 2005-06-20-23.59.59.999999 R99-M9-N9-C:J99-U99 RAS APP FATAL accelerator parity fault on memory controller"
drain_test = Drain(sim_th=best_threshold, max_children=100, max_clusters=None)
for log in logs[:200]:
    drain_test.add_log_message(log)

before_count = len(drain_test.clusters)
cluster_id, anomaly_template, anomaly_change_type = normalize_result(drain_test.add_log_message(anomalous_log))
after_count = len(drain_test.clusters)

print(f"Templates before injection: {before_count}")
print(f"Templates after injection: {after_count}")
print(f"Novel log change type: {anomaly_change_type}")
print(f"Novel log template id: {cluster_id}")
print(f"Novel log template: {anomaly_template}")


TF-IDF matrix shape: (20, 150)
Mean similarity: 0.635
Clusters found above 0.7 similarity: 4

Cluster 1
  - - <*> 2005.07.09 <*> <*> <*> RAS KERNEL INFO generating <*>
  - - <*> <*> <*> <*> <*> RAS KERNEL INFO <*> floating point alignment exceptions
  - - <*> <*> <*> <*> <*> RAS KERNEL INFO CE sym <*> at <*> mask <*>
  - - <*> 2005.07.13 <*> <*> <*> RAS KERNEL INFO generating <*>
  - - <*> 2005.11.04 <*> <*> <*> RAS KERNEL INFO iar <*> dear <*>
  - - <*> 2005.11.03 <*> <*> <*> RAS KERNEL INFO iar <*> dear <*>
  - - <*> 2005.06.05 <*> <*> <*> RAS KERNEL INFO generating <*>
  - - <*> 2005.07.15 <*> <*> <*> RAS KERNEL INFO generating <*>
  - - <*> 2005.06.25 <*> <*> <*> RAS KERNEL INFO generating <*>
  - - <*> 2005.08.26 <*> <*> <*> RAS KERNEL INFO iar <*> dear <*>
  - - <*> 2005.07.23 <*> <*> <*> RAS KERNEL INFO generating <*>
  - - <*> 2005.07.17 <*> <*> <*> RAS KERNEL INFO generating <*>
  - - <*> <*> <*> <*> <*> RAS KERNEL INFO ciod: generated <*> core files for program <*>

Cluster 2

## Phase 4: Mini Log Analyzer and Cross-Dataset Comparison

In [8]:
script_path = SCRIPTS_DIR / "log_analyzer.py"
print(f"Using analyzer script: {script_path}")

bgl_report = subprocess.run([sys.executable, str(script_path), str(BGL_LOG)], capture_output=True, text=True, check=True)
hdfs_report = subprocess.run([sys.executable, str(script_path), str(HDFS_LOG)], capture_output=True, text=True, check=True)

print("BGL analyzer output:")
print(bgl_report.stdout)
print("HDFS analyzer output:")
print(hdfs_report.stdout)

comparison_rows = []
for dataset_name, dataset_path in [("BGL", BGL_LOG), ("HDFS", HDFS_LOG)]:
    dataset_logs = load_lines(dataset_path)
    dataset_drain = Drain(sim_th=best_threshold, max_children=100, max_clusters=None)
    for log in dataset_logs:
        dataset_drain.add_log_message(log)
    comparison_rows.append(
        {
            "dataset": dataset_name,
            "total_logs": len(dataset_logs),
            "unique_templates": len(dataset_drain.clusters),
            "avg_cluster_size": round(len(dataset_logs) / len(dataset_drain.clusters), 2),
        }
    )

df_comparison = pd.DataFrame(comparison_rows).sort_values("unique_templates", ascending=False)
df_comparison.to_csv(RESULTS_DIR / "dataset_comparison.csv", index=False)
print(df_comparison.to_string(index=False))

leader = df_comparison.iloc[0]
lagger = df_comparison.iloc[-1]
print()
print(f"{leader['dataset']} has more templates than {lagger['dataset']} in this run.")
if leader["dataset"] == "BGL":
    print("Reason: BGL_2k contains several distinct failure families and less repetition than HDFS_2k.")
else:
    print("Reason: HDFS_2k contains more message families than BGL_2k in this subset.")


Using analyzer script: C:\Users\Admin\aiops-Thinh\w1\day-b\scripts\log_analyzer.py
BGL analyzer output:

LOG ANALYSIS REPORT

File: C:\Users\Admin\aiops-Thinh\w1\day-b\data\BGL_2k.log
Total lines: 2,000
Unique templates: 151
Avg cluster size: 13.25

TOP 5 TEMPLATES
----------------------------------------------------------------------

1. Template 73
   Count: 180 (9.00%)
   Pattern: - <*> 2005.07.09 <*> <*> <*> RAS KERNEL INFO generating <*>

2. Template 85
   Count: 121 (6.05%)
   Pattern: - <*> <*> <*> <*> <*> RAS KERNEL INFO <*> floating point alignment exceptions

3. Template 2
   Count: 109 (5.45%)
   Pattern: - <*> <*> <*> <*> <*> RAS KERNEL INFO <*> double-hummer alignment exceptions

4. Template 3
   Count: 92 (4.60%)
   Pattern: - <*> <*> <*> <*> <*> RAS KERNEL INFO CE sym <*> at <*> mask <*>

5. Template 77
   Count: 87 (4.35%)
   Pattern: - <*> 2005.07.13 <*> <*> <*> RAS KERNEL INFO generating <*>

SPIKES IN THE LAST HOUR
----------------------------------------------------

## Summary

In [9]:
print("Output files:")
for output_file in [
    RESULTS_DIR / "top_templates.csv",
    RESULTS_DIR / "tuning_results.csv",
    RESULTS_DIR / "template_count_timeseries.png",
    RESULTS_DIR / "dataset_comparison.csv",
]:
    print(f"  - {output_file} ({output_file.stat().st_size:,} bytes)")

print()
print(f"Primary dataset used: {primary_log_path.name}")
print(f"Comparison dataset used: {comparison_log_path.name}")
print(f"Chosen sim_th: {best_threshold}")


Output files:
  - C:\Users\Admin\aiops-Thinh\w1\day-b\results\top_templates.csv (1,075 bytes)
  - C:\Users\Admin\aiops-Thinh\w1\day-b\results\tuning_results.csv (96 bytes)
  - C:\Users\Admin\aiops-Thinh\w1\day-b\results\template_count_timeseries.png (81,752 bytes)
  - C:\Users\Admin\aiops-Thinh\w1\day-b\results\dataset_comparison.csv (94 bytes)

Primary dataset used: BGL_2k.log
Comparison dataset used: HDFS_2k.log
Chosen sim_th: 0.5
